# CNN code

> https://wikidocs.net/217064 

Input   
- [Batch_size, Feature_dimension, Timestamp] 형태   
- timestamp(k)마다 feature dimension(N)이 존재하는 2차원 배열   
- 이 때, feature dimension: 입력값의 임베딩 차원   

Operation (1D convolution)   
- 1차원 시계열 데이터: (Feature dimension, timestamp) 2개의 차원 존재하므로    
- in_channel: feature dimension   
- width: kernel size, length: input channel   
- kernel이 out channel 만큼 생성됨  
    - 이 때, LSTM에 넣기 위해 out channel과 d_model 사이즈를 동일하게 설정해야 함  

Output   
- [Batch_size, Feature_dimension(Channel_dimension), kernel로 변경된 timestamp]   

In [1]:
import torch
import torch.nn as nn

In [ ]:
conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1,
                  dilation=1, groups=1, bias=True, padding_mode='zeros')

conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3)

## CNN

Parameters
- in_channels: input의 feature dimension
- out_channels: output dimension
- kernal_size: 한 kernel당 timestamp 개수
- stride: kernel 이동 크기
- dilation: kernel 내부에서 얼마만큼 띄어서 kernel을 적용할 것인가 (default: 1)
- padding: 한 쪽 방향으로 얼마만큼 padding할 것인가 (그 만큼 양방향으로 적용) (default: 0)
- groups: kernel의 height를 조절
- bias: bias term을 둘 것인가 
- padding_mode: 'zero', 'reflect', 'replicate', 'circular' (default: 'zero')

In [2]:
in_channels = 1
out_channels = 16
kernel_size = 3
stride = 1
dilation = 1
padding = 1
groups = 1
bias = True
padding_mode = 'zeros'
dropout = 0.1

In [ ]:
class CNN(nn.Module):
    def __init__(self,
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                dilation=dilation,
                padding=padding,
                groups=groups,
                bias=bias,
                padding_mode=padding_mode,
                dropout=dropout
    ):
        super(CNN, self).__init__()
        self.cnn = nn.Conv1d(in_channels=in_channels,
                             out_channels=out_channels,
                             kernel_size=kernel_size,
                             stride=stride,
                             dilation=dilation,
                             padding=padding,
                             groups=groups,
                             bias=bias,
                             padding_mode=padding_mode)
        self.relu = nn.ReLU()
        # self.bn = nn.BatchNorm1d(out_channels)
        self.dropout = nn.Dropout(dropout)
        self.flattner = nn.Flatten()

    def forward(self, x):
        x = self.cnn(x)
        x = self.relu(x)
        # x = self.bn(x)
        x = self.dropout(x)
        # x = self.flattner(x)
        x = x.permute(0, 2, 1) # [B, C_out, L_out] -> [B, L_out, C_out] (LSTM에 맞춤)
        return x # [B, L_out, C_out]